# yeast-1-2-8-9_vs_7: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('36_yeast-1-2-8-9_vs_7_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,100.000,50.000,36.298,66.430,0.000
smote,51.194,17.758,12.024,36.667,91.826,64.247,24.967,74.438,34.476
bsmote,51.103,24.117,19.155,35.000,95.094,65.047,25.813,77.387,37.610
ros,62.630,26.661,20.422,43.333,93.839,68.586,26.590,72.980,44.646
adasyn,59.981,17.129,11.324,41.667,87.459,64.563,22.669,72.063,38.555
casb,52.272,35.615,39.636,36.667,98.256,67.461,37.643,75.106,43.943
casb_relaxed,51.375,21.640,19.312,33.333,94.762,64.048,30.942,74.473,36.507
casb_edited,61.667,17.458,11.312,45.000,86.585,65.793,21.597,72.559,39.563
casb_ros,52.109,34.341,36.216,36.667,97.985,67.326,32.447,74.352,43.225


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,16.330,11.429,40.000,6.667,99.891,53.279,28.426,72.132,13.879
smote,66.004,18.423,11.258,53.333,86.151,69.742,15.675,73.445,42.213
bsmote,62.678,22.751,16.013,46.667,91.117,68.892,18.019,76.045,42.714
ros,57.695,19.030,12.798,40.000,91.276,65.638,14.366,71.103,38.362
adasyn,64.597,17.223,10.444,51.667,85.714,68.690,14.715,73.728,40.910
casb,65.110,21.913,14.321,50.000,90.349,70.174,18.966,73.170,43.511
casb_relaxed,65.462,19.569,12.230,51.667,87.894,69.780,20.479,75.234,42.515
casb_edited,71.099,19.188,11.477,61.667,84.132,72.899,12.302,74.363,45.143
casb_ros,57.695,19.030,12.798,40.000,91.276,65.638,15.443,71.181,38.362


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,16.297,10.437,28.333,6.667,99.728,53.197,31.534,79.432,13.367
smote,38.007,17.888,19.786,16.667,97.874,57.270,23.379,76.596,27.948
bsmote,30.006,16.263,23.944,13.333,98.038,55.685,23.982,80.301,23.134
ros,41.609,24.407,28.083,23.333,98.311,60.822,26.063,78.401,33.008
adasyn,39.648,18.998,20.786,18.333,97.764,58.049,19.766,78.983,29.323
casb,44.112,27.595,45.000,23.333,98.856,61.095,28.867,80.059,35.854
casb_relaxed,29.990,15.689,21.012,13.333,97.928,55.631,23.429,78.712,22.840
casb_edited,44.921,17.571,15.586,21.667,95.856,58.761,19.938,79.487,31.246
casb_ros,41.705,26.253,39.369,23.333,98.746,61.040,30.178,79.760,33.979


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,16.330,11.429,40.000,6.667,99.891,53.279,22.665,80.574,13.879
smote,71.777,20.325,12.435,61.667,84.731,73.199,21.921,80.806,46.051
bsmote,69.088,20.438,12.679,56.667,86.858,71.762,21.872,82.253,44.763
ros,70.075,20.591,12.877,60.000,84.622,72.311,23.222,80.780,45.333
adasyn,69.644,18.202,10.948,60.000,83.370,71.685,21.438,80.481,43.923
casb,66.062,19.207,11.998,51.667,87.133,69.400,21.267,80.692,42.635
casb_relaxed,70.450,19.968,12.150,60.000,85.827,72.914,21.401,80.872,45.209
casb_edited,72.942,19.736,11.789,65.000,83.051,74.025,21.611,80.627,46.339
casb_ros,64.455,17.878,11.123,51.667,86.156,68.911,21.344,80.639,41.166


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,12.239,8.323,27.083,5.000,99.878,52.439,29.731,74.642,10.281
smote,56.745,18.599,13.876,42.083,90.146,66.114,21.485,76.321,37.672
bsmote,53.218,20.892,17.948,37.917,92.777,65.347,22.421,78.996,37.055
ros,58.002,22.672,18.545,41.667,92.012,66.839,22.560,75.816,40.337
adasyn,58.468,17.888,13.375,42.917,88.577,65.747,19.647,76.314,38.178
casb,56.889,26.083,27.739,40.417,93.648,67.032,26.686,77.257,41.486
casb_relaxed,54.319,19.217,16.176,39.583,91.603,65.593,24.063,77.323,36.768
casb_edited,62.657,18.488,12.541,48.333,87.406,67.870,18.862,76.759,40.573
casb_ros,53.991,24.375,24.877,37.917,93.541,65.729,24.853,76.483,39.183


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,61.667,ros,62.630,-0.963,LOSS
1,f1,casb,35.615,ros,26.661,8.954,WIN
2,precision,casb,39.636,ros,20.422,19.214,WIN
3,recall,casb_edited,45.000,ros,43.333,1.667,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,71.099,smote,66.004,5.095,WIN
1,f1,casb,21.913,bsmote,22.751,-0.838,LOSS
2,precision,casb,14.321,bsmote,16.013,-1.692,LOSS
3,recall,casb_edited,61.667,smote,53.333,8.333,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,44.921,ros,41.609,3.312,WIN
1,f1,casb,27.595,ros,24.407,3.188,WIN
2,precision,casb,45.000,ros,28.083,16.917,WIN
3,recall,casb,23.333,ros,23.333,0.000,TIE


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,62.657,adasyn,58.468,4.190,WIN
1,f1,casb,26.083,ros,22.672,3.410,WIN
2,precision,casb,27.739,ros,18.545,9.194,WIN
3,recall,casb_edited,48.333,adasyn,42.917,5.417,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.237324,0.507246,0.000000,None
1,42,1,svm,smote,0.75,0.541736,0.133333,0.083333,0.333333,0.880435,0.606884,0.126847,0.691123,0.337535,None
2,42,1,svm,bsmote,0.50,0.566262,0.266667,0.222222,0.333333,0.961957,0.647645,0.245353,0.679348,0.416464,None
3,42,1,svm,ros,0.75,0.556581,0.190476,0.133333,0.333333,0.929348,0.631341,0.136091,0.597826,0.373529,None
4,42,1,svm,adasyn,0.75,0.546729,0.148148,0.095238,0.333333,0.896739,0.615036,0.130508,0.657609,0.347439,None
5,42,1,svm,casb,0.50,0.407137,0.250000,0.500000,0.166667,0.994565,0.580616,0.243864,0.597826,0.328569,None
6,42,1,svm,casb_relaxed,0.50,0.407137,0.250000,0.500000,0.166667,0.994565,0.580616,0.237082,0.612319,0.328569,None
7,42,1,svm,casb_edited,0.75,0.543406,0.137931,0.086957,0.333333,0.885870,0.609601,0.097245,0.664855,0.340668,None
8,42,1,svm,casb_ros,0.50,0.406023,0.222222,0.333333,0.166667,0.989130,0.577899,0.242362,0.605072,0.314123,None
9,42,1,knn,original,0.50,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.208553,0.619565,0.000000,None
